# ¿La música se volvió más enérgica y menos alegre? Argentina frente al mundo (1920–2021)

**TP Grupal · Laboratorio de Métodos Cuantitativos · FCE UBA · 2C 2026**

**Integrantes:** Sergio Sebastian Cordoba (921765) · Mauro Nohara Gushiken (922170) · Facundo Malpeli (921871) · Manuel Mariani (921739)

**Dataset:** [Spotify Dataset 1921–2020, 600k+ Tracks](https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks) (Kaggle, usuario *yamaerenay*). Usamos dos archivos:
- `tracks.csv`: una fila por canción, con su fecha de lanzamiento y sus características de audio.
- `artists.csv`: los artistas con los géneros que les asigna Spotify. Lo usamos para identificar la música argentina.

### Uso de IA
Las celdas marcadas con **[Asistido por IA]** se armaron con ayuda de Claude, en un proyecto que tenía como contexto las pautas del TP. Prompts utilizados:
- Sección 3.1: «como puedo explorar el dataset»
- Secciones 3.2 a 3.4: «¿La música se volvió más enérgica y menos alegre entre 1920 y 2021? ¿La música argentina siguió la misma tendencia que la global? podrías hacer el trabajo en base a eso? y adaptar lo que ya teníamos?»
- *(Agregar acá los prompts que se usen para las interpretaciones, si se usa IA.)*

## 3.1 Carga y exploración inicial

### a) Carga del dataset · [Asistido por IA]

Descargamos el dataset directamente desde Kaggle con la librería `kagglehub`, para que cualquiera pueda volver a correr el notebook sin subir archivos a mano.

> Si la descarga falla, se puede bajar el ZIP desde la página de Kaggle (requiere cuenta), subir `tracks.csv` y `artists.csv` a Colab y cargarlos con `pd.read_csv("tracks.csv")` y `pd.read_csv("artists.csv")`.

In [ ]:
%pip install -q kagglehub

In [ ]:
import os
import ast
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

# Descarga el dataset y devuelve la carpeta donde quedaron los archivos
path = kagglehub.dataset_download("yamaerenay/spotify-dataset-19212020-600k-tracks")
print("Archivos descargados:", os.listdir(path))

df = pd.read_csv(os.path.join(path, "tracks.csv"))        # canciones
artists = pd.read_csv(os.path.join(path, "artists.csv"))  # artistas y sus géneros

# Colores fijos para todo el trabajo
COLORES = {"Argentina": "#2a78d6", "Resto": "#eb6834"}

#### Primeras y últimas filas
Miramos el principio y el final de la tabla para ver cómo vienen los datos: qué tiene cada columna, ejemplos de valores y si hay algo raro a simple vista.

In [ ]:
display(df.head())
display(df.tail())

### b) Estructura: dimensiones, columnas y tipos de dato · [Asistido por IA]

Detalles a tener en cuenta:
- `release_date` es **texto**, no fecha: hay que convertirla para trabajar con años.
- `artists` e `id_artists` son textos con forma de lista, por ejemplo `"['Uli']"`.
- `explicit` y `mode` son 0/1, y `key` es un número de 0 a 11 que representa la nota (0 = Do).

In [ ]:
print("Filas (canciones):", df.shape[0])
print("Columnas:", df.shape[1])
df.dtypes

#### Cobertura temporal
`release_date` no tiene siempre la misma precisión: algunas canciones solo tienen el año, otras año y mes, y otras la fecha completa. Contamos cuántas hay de cada tipo según la cantidad de caracteres y tomamos los primeros 4 para obtener el año.

El año se calcula acá solo para explorar; la columna definitiva se crea en la sección 3.3. El dataset **no tiene país**: la música argentina se identifica en la sección 3.3 a partir de los géneros de cada artista.

In [ ]:
# 4 caracteres = solo año, 7 = año y mes, 10 = fecha completa
precision = df["release_date"].str.len().map({4: "solo año", 7: "año y mes", 10: "fecha completa"})
print(precision.value_counts(dropna=False))

anio = df["release_date"].str[:4].astype(int)   # temporal, solo para explorar
print()
print("Año mínimo:", anio.min())
print("Año máximo:", anio.max())

Contamos las canciones por década para ver si el dataset está balanceado en el tiempo.

In [ ]:
decada = (anio // 10) * 10
decada.value_counts().sort_index().rename("canciones")

Lo graficamos por año. Si las primeras décadas tienen pocas canciones, sus promedios van a ser menos confiables.

In [ ]:
anio.value_counts().sort_index().plot(figsize=(10, 4), color="#2a78d6")
plt.title("Cantidad de canciones por año de lanzamiento")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Cantidad de canciones")
plt.grid(alpha=0.3)
plt.show()

#### El archivo de artistas
Revisamos cuántos artistas hay y qué proporción no tiene ningún género asignado, porque de los géneros depende identificar la música argentina.

In [ ]:
print("Artistas:", artists.shape[0])
sin_genero = (artists["genres"] == "[]").mean() * 100
print(f"Artistas sin género asignado: {sin_genero:.1f} %")
artists.head()

### c) Calidad de los datos · [Asistido por IA]

#### Valores nulos

In [ ]:
df.isnull().sum()

#### Duplicados
Los revisamos en tres niveles:
1. **Filas idénticas.**
2. **IDs repetidos.**
3. **Misma canción y artista** con distinto `id`: suelen ser reediciones, remasters o compilados. Importan para nuestra pregunta, porque una canción de 1965 reeditada en 2010 aparece con fecha de 2010.

In [ ]:
print("Filas completamente duplicadas:", df.duplicated().sum())
print("IDs repetidos:", df["id"].duplicated().sum())
print("Misma canción y artista repetidos:", df.duplicated(subset=["name", "artists"]).sum())

In [ ]:
# Las 5 combinaciones canción + artista que más se repiten
repetidas = df[df.duplicated(subset=["name", "artists"], keep=False)]
repetidas.groupby(["name", "artists"]).size().sort_values(ascending=False).head(5)

#### Estadísticas descriptivas
Resumen de las columnas numéricas, transpuesto (`.T`) para que cada variable quede en una fila.

In [ ]:
df.describe().T.round(2)

#### Valores sospechosos
- `tempo = 0` o `time_signature = 0`: Spotify no pudo medir el ritmo.
- `speechiness > 0.66`: según Spotify, es audio casi todo hablado (audiolibros, poesía, discursos).
- Duraciones extremas.

In [ ]:
print("Canciones con tempo = 0:", (df["tempo"] == 0).sum())
print("Canciones con time_signature = 0:", (df["time_signature"] == 0).sum())
print("Canciones con popularidad = 0:", (df["popularity"] == 0).sum())
print("Grabaciones casi solo habladas (speechiness > 0.66):", (df["speechiness"] > 0.66).sum())
print("Canciones de más de 20 minutos:", (df["duration_ms"] > 20 * 60 * 1000).sum())
print("Canciones de menos de 30 segundos:", (df["duration_ms"] < 30 * 1000).sum())

In [ ]:
# Ejemplos de las grabaciones más "habladas": ¿son música o no?
(df[df["speechiness"] > 0.66]
   .sort_values("speechiness", ascending=False)
   [["name", "artists", "release_date", "speechiness"]]
   .head(10))

### Hallazgos de la exploración · [Asistido por IA] *(revisar)*

- **Tamaño:** 586.672 canciones y 20 columnas.
- **Cobertura temporal:** de 1900 a 2021, pero 1900 tiene una sola canción y la década de 2020 solo incluye 2020 y 2021. La cantidad de canciones crece mucho con el tiempo (7.610 en los años 20 contra 108.875 en los 90), así que los promedios de las primeras décadas son menos estables.
- **Fechas:** el 23,3 % de las canciones solo tiene el año (136.489), por eso trabajamos con el año y no con la fecha completa.
- **Nulos:** solo en `name` (71 canciones, 0,01 %). No afectan a las variables de audio.
- **Duplicados:** no hay filas ni IDs repetidos, pero 60.062 canciones (10,2 %) repiten nombre y artista: son reediciones y compilados que pueden tener fechas posteriores a la original.
- **Valores raros:** 22.598 grabaciones casi solo habladas (3,9 %). Las más extremas son audiolibros en alemán fechados en 1952 y 1960, el año de publicación del libro y no el de la grabación, lo que muestra que la fecha no siempre es confiable. También hay 328 canciones con tempo 0 y 228 de menos de 30 segundos.
- **Artistas:** *(completar con el % sin género que muestra la celda de artistas)*.
- **Decisiones para la sección 3.3:** trabajar desde 1920, excluir grabaciones habladas y muy cortas, eliminar reediciones y cruzar con los artistas para identificar la música argentina.

## 3.2 Pregunta de investigación · [Asistido por IA]

**a) ¿Cuál es la pregunta?**

¿La música se volvió más enérgica y menos alegre entre 1920 y 2021? ¿La música argentina siguió la misma tendencia que la global?

Para responderla usamos dos atributos que Spotify calcula para cada canción:
- **`energy`** (energía, de 0 a 1): qué tan intensa y activa suena la canción.
- **`valence`** (valencia, de 0 a 1): qué tan positiva o alegre suena. Valores altos son canciones alegres; valores bajos, tristes o tensas.

**b) ¿Por qué vale la pena responderla?**

La música refleja los cambios culturales y tecnológicos de cada época. Estudios previos encontraron que la música popular se volvió más negativa en las últimas décadas (Brand, Acerbi y Mesoudi, 2019; Interiano et al., 2018). Esta pregunta permite contrastar esa tendencia a lo largo de un siglo y analizar si la música argentina se fue pareciendo a la global o si conserva un perfil propio. La respuesta es útil para la industria musical, porque ayuda a decidir qué producir y para qué público, y para el diseño de políticas culturales de fomento de la música nacional.

**c) ¿Qué respuesta encontraron?**

*(Completar al final del análisis, con los números de las secciones 3.3 y 3.4.)*

## 3.3 Transformaciones y resumen estadístico

### a) Transformaciones justificadas · [Asistido por IA]

1. **Año y década de lanzamiento.** `release_date` mezcla formatos, así que tomamos los primeros 4 caracteres como año y agrupamos por década, que suaviza el ruido de los años con pocas canciones.
2. **Filtro de filas.** Excluimos:
   - las canciones anteriores a 1920 (en 1900 hay una sola, que distorsionaría los promedios);
   - las grabaciones casi solo habladas (`speechiness > 0.66`), porque no son música;
   - las pistas de menos de 30 segundos (intros, separadores);
   - las 71 canciones sin nombre, que no se pueden controlar como duplicadas.
3. **Eliminación de reediciones.** Cuando una canción aparece varias veces con el mismo nombre y artista, nos quedamos con la de fecha más antigua, que es la más cercana al lanzamiento original.
4. **Origen de la música (Argentina / Resto).** Cruzamos cada canción con su artista principal en `artists.csv` y la marcamos como argentina si alguno de sus géneros es argentino.

In [ ]:
# 1) Año y década
df["year"] = df["release_date"].str[:4].astype(int)
df["decada"] = (df["year"] // 10) * 10

# 2) Filtro de filas
df_analisis = df[
    (df["year"] >= 1920)
    & (df["speechiness"] <= 0.66)
    & (df["duration_ms"] >= 30_000)
    & (df["name"].notna())
].copy()
print(f"Canciones originales:        {len(df):,}")
print(f"Después de filtrar:          {len(df_analisis):,}")

# 3) Reediciones: nos quedamos con la versión más antigua de cada canción + artista
antes = len(df_analisis)
df_analisis = (df_analisis.sort_values("year")
                          .drop_duplicates(subset=["name", "artists"], keep="first")
                          .copy())
print(f"Reediciones eliminadas:      {antes - len(df_analisis):,}")
print(f"Canciones para el análisis:  {len(df_analisis):,}")

#### Identificación de la música argentina
Spotify asigna géneros a los artistas, no a las canciones. Usamos los géneros del **artista principal** (el primero de `id_artists`) y consideramos argentina una canción si alguno de esos géneros contiene alguna de estas palabras: *argentin, tango, rock nacional, cuarteto, chamame, cumbia villera, rkt*. Dejamos afuera "finnish tango" (tango finlandés) y "rock nacional brasileiro".

In [ ]:
artists["generos"] = artists["genres"].apply(ast.literal_eval)

# Artista principal de cada canción y sus géneros
df_analisis["id_artista"] = df_analisis["id_artists"].apply(lambda x: (ast.literal_eval(x) or [None])[0])
df_analisis = df_analisis.merge(
    artists[["id", "generos"]].rename(columns={"id": "id_artista"}),
    on="id_artista", how="left")
df_analisis["generos"] = df_analisis["generos"].apply(lambda g: g if isinstance(g, list) else [])

# 4) Origen
CLAVES_ARG = ["argentin", "tango", "rock nacional", "cuarteto", "chamame", "cumbia villera", "rkt"]
EXCLUIR = ["finnish", "brasileiro"]

def es_argentina(generos):
    return any(any(c in g for c in CLAVES_ARG) and not any(e in g for e in EXCLUIR) for g in generos)

df_analisis["origen"] = np.where(df_analisis["generos"].apply(es_argentina), "Argentina", "Resto")
df_analisis["origen"].value_counts()

**Control:** listamos los géneros que entraron como argentinos para verificar que no se haya colado ninguno de otro país. Si aparece alguno, se agrega a `EXCLUIR` y se vuelve a correr la celda anterior.

In [ ]:
generos_arg = Counter(
    g for lista in df_analisis.loc[df_analisis["origen"] == "Argentina", "generos"]
    for g in lista if any(c in g for c in CLAVES_ARG))
pd.Series(dict(generos_arg.most_common(30)), name="canciones")

**Cantidad de canciones por década y origen.** Para que un promedio sea confiable pedimos un mínimo de 100 canciones; las décadas que no lleguen quedan fuera de las comparaciones. La década de 2020 solo tiene 2020 y 2021, así que para medir cambios usamos como última década **2010**, la última completa.

In [ ]:
MIN_CANCIONES = 100
DECADA_FINAL = 2010   # última década completa

conteo = df_analisis.groupby(["decada", "origen"]).size().unstack(fill_value=0)
display(conteo)

# Primera década en que Argentina tiene suficientes canciones: es el punto de partida de la comparación
DECADA_INICIAL = int(conteo.index[conteo["Argentina"] >= MIN_CANCIONES].min())
print(f"Comparamos desde los {DECADA_INICIAL} hasta los {DECADA_FINAL}.")

### b) Estadísticos agrupados con `.groupby()` · [Asistido por IA]

**Resumen 1: evolución global por década.** Promedio de energía, valence, acusticidad y bailabilidad de todas las canciones de cada década.

In [ ]:
ATRIBUTOS = ["energy", "valence", "acousticness", "danceability"]
NOMBRES = {"energy": "Energía", "valence": "Valence (alegría)",
           "acousticness": "Acusticidad", "danceability": "Bailabilidad"}

resumen_global = df_analisis.groupby("decada").agg(
    canciones=("energy", "size"),
    **{a: (a, "mean") for a in ATRIBUTOS}
).round(3)
resumen_global

**Resumen 2: Argentina frente al resto, por década.** Los mismos promedios separados por origen. Las décadas con menos de 100 canciones aparecen vacías.

In [ ]:
por_origen = df_analisis.groupby(["decada", "origen"]).agg(
    canciones=("energy", "size"),
    **{a: (a, "mean") for a in ATRIBUTOS}
).reset_index()
por_origen.loc[por_origen["canciones"] < MIN_CANCIONES, ATRIBUTOS] = np.nan

por_origen.pivot(index="decada", columns="origen", values=["canciones", "energy", "valence"]).round(3)

**Resumen 3: cuánto cambió cada atributo.** Diferencia entre la década inicial y la final, para cada origen. Un valor positivo indica que el atributo aumentó.

In [ ]:
inicio = por_origen[por_origen["decada"] == DECADA_INICIAL].set_index("origen")[ATRIBUTOS]
final = por_origen[por_origen["decada"] == DECADA_FINAL].set_index("origen")[ATRIBUTOS]
cambios = (final - inicio).T.rename(index=NOMBRES).round(3)

print(f"Cambio entre los {DECADA_INICIAL} y los {DECADA_FINAL} (valor final − valor inicial):")
cambios

**Interpretación de los resúmenes** *(completar después de ejecutar)*

- **Tendencia global:** la energía pasó de ___ en los años 20 a ___ en los 2010, y la valence de ___ a ___. ¿La música se volvió más enérgica? ¿Y menos alegre? ¿Los dos cambios ocurren al mismo tiempo o en épocas distintas?
- **Argentina:** ¿la energía y la valence argentinas se movieron en la misma dirección que el resto? ¿El cambio fue más grande o más chico (ver Resumen 3)?
- **Década clave:** ¿en qué década se ve el salto más grande? ¿Qué cambios tecnológicos o culturales coinciden con ese momento?

## 3.4 Análisis gráfico · [Asistido por IA]

### Gráfico 1: evolución temporal de la energía y la alegría
Dos paneles con la misma escala: la energía a la izquierda y la valence a la derecha, cada uno con una línea para la música argentina y otra para el resto.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, var in zip(ejes, ["energy", "valence"]):
    for origen in ["Argentina", "Resto"]:
        datos = por_origen[por_origen["origen"] == origen]
        ax.plot(datos["decada"], datos[var], marker="o", linewidth=2,
                color=COLORES[origen], label=origen)
    ax.set_title(NOMBRES[var])
    ax.set_xlabel("Década de lanzamiento")
    ax.set_ylim(0, 1)
    ax.grid(color="#e1e0d9")
    ax.set_axisbelow(True)
ejes[0].set_ylabel("Promedio (0 a 1)")
ejes[0].legend()
fig.suptitle("¿La música se volvió más enérgica y menos alegre? Argentina vs. resto del mundo")
plt.tight_layout()
plt.savefig("grafico1_energia_valence_por_decada.png", dpi=300, bbox_inches="tight")
plt.show()

**Interpretación** *(completar)*: ¿cuándo empieza a subir la energía? ¿La valence baja en todo el período o solo en algunas décadas? ¿La línea argentina va adelantada, atrasada o a la par del resto? ¿Qué significa eso para la pregunta?

### Gráfico 2: cuánto cambió cada atributo, Argentina vs. resto
Barras con la diferencia entre la década inicial y la de 2010 para cada atributo. Si las barras de los dos colores van en la misma dirección y tienen un tamaño parecido, la música argentina siguió la tendencia global.

In [ ]:
x = np.arange(len(cambios))
ancho = 0.36
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.19, cambios["Argentina"], ancho, color=COLORES["Argentina"], label="Argentina")
ax.bar(x + 0.19, cambios["Resto"], ancho, color=COLORES["Resto"], label="Resto")
ax.axhline(0, color="#898781", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(cambios.index)
ax.set_title(f"Cambio de cada atributo entre los {DECADA_INICIAL} y los {DECADA_FINAL}")
ax.set_xlabel("Atributo de audio")
ax.set_ylabel("Cambio en el promedio (escala 0 a 1)")
ax.legend()
ax.grid(axis="y", color="#e1e0d9")
ax.set_axisbelow(True)
plt.savefig("grafico2_cambio_por_atributo.png", dpi=300, bbox_inches="tight")
plt.show()

**Interpretación** *(completar)*: ¿qué atributos subieron y cuáles bajaron? ¿La música argentina cambió en la misma dirección que el resto? ¿En cuál de los dos el cambio fue más fuerte?

### Gráfico 3: mapa de ánimo (relación entre energía y valence)
Cruzamos las dos variables de la pregunta. Cada punto es el promedio de una década y la línea muestra el recorrido en el tiempo. Los cuadrantes ayudan a leer el "ánimo" de la música: alegre y enérgica, tensa, triste o tranquila.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for origen in ["Argentina", "Resto"]:
    datos = por_origen[por_origen["origen"] == origen].dropna(subset=["valence", "energy"])
    ax.plot(datos["valence"], datos["energy"], marker="o", linewidth=2,
            color=COLORES[origen], label=origen)
    for _, fila in datos.iloc[[0, -1]].iterrows():   # rotulamos la primera y la última década
        ax.annotate(f"{int(fila['decada'])}s", (fila["valence"], fila["energy"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9, color="#52514e")

ax.axvline(0.5, color="#c3c2b7", linewidth=1)
ax.axhline(0.5, color="#c3c2b7", linewidth=1)
for x_txt, y_txt, texto in [(0.98, 0.98, "Alegre y enérgica"), (0.02, 0.98, "Tensa / agresiva"),
                            (0.02, 0.02, "Triste / melancólica"), (0.98, 0.02, "Tranquila / relajada")]:
    ax.text(x_txt, y_txt, texto, transform=ax.transAxes,
            ha="right" if x_txt > 0.5 else "left", va="top" if y_txt > 0.5 else "bottom",
            fontsize=9, color="#898781")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Mapa de ánimo de la música por década: Argentina vs. resto")
ax.set_xlabel("Valence (0 = triste, 1 = alegre)")
ax.set_ylabel("Energía (0 = calma, 1 = intensa)")
ax.legend(loc="center right")
plt.savefig("grafico3_mapa_de_animo.png", dpi=300, bbox_inches="tight")
plt.show()

**Interpretación** *(completar)*: ¿de qué cuadrante a qué cuadrante se movió la música? ¿Los dos recorridos son parecidos? ¿Alguno terminó en la zona "tensa" (enérgica pero poco alegre)?

## Limitaciones · [Asistido por IA] *(revisar)*

- **Fechas:** aun eliminando reediciones, algunas canciones conservan la fecha de una edición posterior (ver los audiolibros fechados con el año del libro).
- **Atributos estimados:** energy y valence son estimaciones de un algoritmo de Spotify, no mediciones. Valence es la más subjetiva.
- **Tecnología de grabación:** las grabaciones antiguas (mono, con ruido, de bajo volumen) pueden medir menos energía por la tecnología y no solo por la música.
- **Música argentina:** se define por los géneros del artista, que Spotify asignó hoy. Quedan afuera los artistas sin género y los argentinos etiquetados con géneros genéricos (por ejemplo, "latin pop"); el tango uruguayo cuenta como argentino.
- **Catálogo:** lo que está en Spotify no representa toda la música de cada época, sobre todo en las primeras décadas.

## Conclusión
*(Completar al final: responder la pregunta en 3 o 4 oraciones, con los números principales, y copiar la respuesta en la sección 3.2 c.)*